# HEL calculator — independent replication

**Layer 5 of the validation campaign (Package 5).**

This notebook reproduces the calculator's most consequential physics outputs from first-principles formulas using only `numpy` and `scipy`. **No module under `physics/` is imported.** The independent results are then compared against the calculator's outputs at the three canonical scenarios pinned in `tests/golden/`.

**Target:** ≤ 5% agreement on every comparison row — directional confirmation the calculator is in the right neighborhood, not bit-for-bit reproduction.

**Modules replicated:**
- M1 — Gaussian-beam fundamentals (closed form)
- M5 — spherical-wave Fried r₀ on the HV-5/7 Cn² profile + engineering w_turb
- M7 — exact-Gaussian w_diff + quadrature-sum spot + I_peak
- M8 — 1-D transient heat PDE (independent explicit-FD code)
- M9 — ANSI Z136.1-2014 piecewise MPE + top-hat / Gaussian-peak NOHD

**Modules deliberately NOT replicated** (out of Layer 5 scope per the campaign plan): M2 (P_exit = η·P0, trivial), M3 (geometry trig — Package 1 covered it), M4 (atmosphere — same McClatchey table, replication adds no signal), M6 thermal blooming (engineering 4√2 prefactor and 0.3 broadening — independent code hits the same engineering choices), M10 (power/thermal arithmetic — Package 2 cross-module test), M11 (validation runner).

**Implementations live in** `validation/replication/replication.py` so the same code can run from the notebook or from the command line (`python validation/replication/replication.py`).

In [1]:
import math
import sys
from pathlib import Path

# Locate the repo root from the notebook's location.
REPO = Path('.').resolve()
while not (REPO / 'physics').is_dir() and REPO.parent != REPO:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'validation' / 'replication'))
sys.path.insert(0, str(REPO))

from replication import (
    m1_independent,
    m5_independent,
    m7_independent,
    m8_independent,
    m9_independent,
    replicate_scenario,
)
from tests.golden.scenarios import SCENARIOS

print(f'Repo root: {REPO}')
print(f'Scenarios: {list(SCENARIOS)}')

Repo root: C:\Users\lenovo\Documents\claude\HEL PROJECT
Scenarios: ['c_uas_1500m', 'counter_rocket_3000m', 'long_range_8000m']


## Independent M1 — Gaussian-beam fundamentals

Per Siegman 1986 §17 and SPEC §3 M1:

$$ w_0 = \frac{D}{2}, \quad \theta_\text{diff} = M^2 \cdot \frac{4\lambda}{\pi D}, \quad z_R = \frac{\pi w_0^2}{\lambda}, \quad I_\text{exit} = \frac{2 P_0}{\pi w_0^2} $$

Note `theta_diff` is full-angle (project's convention).

In [2]:
scen = SCENARIOS['c_uas_1500m']
out = m1_independent(scen['P0'], scen['M2'], scen['D'], scen['wavelength'])
for k, v in out.items():
    print(f'  {k:12s} = {v:.6g}')

  w0           = 0.05
  theta_diff   = 1.63484e-05
  zR           = 7340.17
  I_exit       = 763944


## Independent M5 — spherical-wave r₀ over HV-5/7

Per Andrews & Phillips 2005 §6.5 and CLAUDE §7.1:

$$ r_0^\text{sph} = \left(0.423 \cdot k^2 \cdot \int_0^L C_n^2(z(s)) \cdot (s/L)^{5/3} \, ds\right)^{-3/5} $$

$$ w_\text{turb} = \frac{2L}{k \cdot r_0^\text{sph}} $$

HV-5/7 profile (Hufnagel 1974, Valley 1980):

$$ C_n^2(h) = 0.00594 \cdot (v/27)^2 \cdot (10^{-5} h)^{10} \cdot e^{-h/1000} + 2.7 \times 10^{-16} \cdot e^{-h/1500} + C_{n,\text{ground}}^2 \cdot e^{-h/100} $$

Slant-path altitude is linearly interpolated between H_e (s=0) and H_t (s=L). The (s/L)^(5/3) factor vanishes at s=0 so the integrand is well-behaved; `scipy.integrate.quad` handles it adaptively.

In [3]:
import json
scen = SCENARIOS['c_uas_1500m']
with open(REPO / 'tests' / 'golden' / 'c_uas_1500m.json') as f:
    golden = json.load(f)
out = m5_independent(
    L=golden['R_slant'], wavelength=scen['wavelength'],
    v_HV=scen['v_HV'], Cn2_ground=scen['Cn2_ground'],
    H_e=scen['H_e'], H_t=scen['H_t'],
)
for k, v in out.items():
    print(f'  {k:18s} = {v:.6g}')

  Cn2_integrated     = 2.57192e-12
  r0_sph             = 0.113628
  w_turb             = 0.00449613


## Independent M7 — exact-Gaussian spot, quadrature sum, peak irradiance

Per CLAUDE §7.1 invariants and SPEC §3 M7:

$$ w_\text{diff}(L) = w_0 \cdot \sqrt{1 + \left(\frac{M^2 L}{z_R}\right)^2}, \quad w_\text{jit} = 2 \sigma_\text{jit} L $$

$$ w_\text{total}^2 = w_\text{diff}^2 + w_\text{turb}^2 + w_\text{jit}^2 + w_\text{bloom}^2 $$

$$ I_\text{peak} = \frac{2 P_\text{exit} \tau_\text{atm} S_\text{TB}}{\pi w_\text{total}^2} $$

M2/M4/M6 outputs (P_exit, τ_atm, S_TB, w_bloom) come from the calculator since those modules are not in Layer 5 scope. The diffraction term is the **exact** Gaussian, not the far-field asymptote — CLAUDE §7.1 invariant.

## Independent M8 — 1-D transient heat PDE

Independent explicit finite-difference solver. Front face: surface flux
$q_\text{in} = A_\lambda \cdot I_\text{aim} - \varepsilon \sigma (T^4 - T_\text{amb}^4)$.
Back face: convective $q_\text{back} = h_\text{conv}(T - T_\text{amb})$ with $h_\text{conv} = 10 + 6.2\sqrt{v_\text{tgt}}$. Failure at $T_\text{surface} \geq T_\text{fail}$.

Grid: 50 µm spacing, ≥21 nodes (matches project's choice). CFL: $\Delta t = 0.4 \Delta x^2 / (2\alpha)$ explicit-stable.

## Independent M9 — ANSI Z136.1 MPE + NOHD

Per ANSI Z136.1-2014 (no C_A applied per SPEC §10.3 conservative disposition):
- Band A (400–1400 nm), 18 µs ≤ t ≤ 10 s: $\text{MPE} = 1.8 \times 10^{-3} \cdot t^{-0.25}$ W/cm²
- Band B (1400–4000 nm), t ≤ 10 s: $\text{MPE} = 0.56 \cdot t^{-0.75}$ W/cm²
- Pulsed (t < 18 µs): $\text{MPE} = 5 \times 10^{-7} / t$ W/cm² (per v1.12 paired SPEC + code edit, ANSI Z136.1-2014 Table 5a single-pulse retinal radiant exposure)

$$ \text{NOHD}_\text{tophat} = \max\!\left(0, \frac{1}{\theta} \sqrt{\frac{4 P_0}{\pi \cdot \text{MPE}}} - \frac{D}{\theta}\right) $$

$$ \text{NOHD}_\text{gausspeak} = \max\!\left(0, \frac{1}{\theta} \sqrt{\frac{8 P_0}{\pi \cdot \text{MPE}}} - \frac{D}{\theta}\right) $$

$D/\theta$ is the aperture correction (the beam fills the aperture out to that range).

## Run all three scenarios and tabulate

In [4]:
results = []
for name, scen in SCENARIOS.items():
    golden_file = REPO / 'tests' / 'golden' / f'{name}.json'
    results.append(replicate_scenario(name, scen, golden_file))

for scenario in results:
    print(f'\n=== {scenario["scenario"]} ===')
    print(f'{"Module":<8}{"Key":<18}{"Independent":>22}{"Calculator":>22}{"rel err %":>10}')
    print('-' * 80)
    for row in scenario['rows']:
        ind = row['independent']
        cal = row['calculator']
        err = row['rel_err_pct']
        ind_s = ind if isinstance(ind, str) else (f'{ind:.4g}' if math.isfinite(ind) else str(ind))
        cal_s = cal if isinstance(cal, str) else (f'{cal:.4g}' if math.isfinite(cal) else str(cal))
        err_s = f'{err:+.2f}' if math.isfinite(err) else 'n/a'
        print(f'{row["module"]:<8}{row["key"]:<18}{ind_s:>22}{cal_s:>22}{err_s:>10}')

abs_errs = [abs(r['rel_err_pct']) for s in results for r in s['rows'] if math.isfinite(r['rel_err_pct'])]
print(f'\nWorst |rel err|: {max(abs_errs):.2f}% across {sum(len(s["rows"]) for s in results)} comparisons.')


=== c_uas_1500m ===
Module  Key                          Independent            Calculator rel err %
--------------------------------------------------------------------------------
M1      theta_diff                     1.635e-05             1.635e-05     +0.00
M1      w0                                  0.05                  0.05     +0.00
M1      zR                                  7340                  7340     +0.00
M1      I_exit                         7.639e+05             7.639e+05     +0.00
M5      Cn2_integrated                 2.572e-12             2.572e-12     +0.00
M5      r0_sph                            0.1136                0.1136     +0.00
M5      w_turb                          0.004496              0.004496     +0.00
M7      w_diff                           0.05148               0.05148     +0.00
M7      w_jit                               0.03                  0.03     +0.00
M7      w_total                          0.06008               0.06021     -0.23
M7     

## Acceptance

Layer 5 target: ≤ 5% agreement across all rows. **Achieved: 4.04% worst-case** (see `replication_report.md` for the analysis of where the residual comes from).